# SYE6105 项目（数据集 A：NASA C-MAPSS FD001–FD004）

本 Notebook 进行**专业化、完整化**交付，覆盖 `dataset_A.zip` 中全部子集（FD001–FD004）：
- 分类任务：30 周期内失效风险识别（Classification）
- 回归任务：RUL 连续预测（Regression）
- 每个任务至少两种算法并在四个子集上统一对比
- 专业可视化：趋势图、混淆矩阵、PR 曲线、校准曲线、特征重要性、回归误差分布


## 1. 依赖、随机种子与中文字体设置
- 全流程可复现：固定随机种子。
- 为避免 matplotlib 中文缺字（missing glyph）问题，设置 CJK 字体优先级和自动回退。


In [ ]:
import io
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    recall_score,
    average_precision_score,
    roc_auc_score,
    precision_score,
    f1_score,
    brier_score_loss,
    confusion_matrix,
    ConfusionMatrixDisplay,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    precision_recall_curve,
)
from sklearn.calibration import calibration_curve
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    GradientBoostingRegressor,
)
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42
N_HORIZON = 30
SUBSETS = ['FD001', 'FD002', 'FD003', 'FD004']
ROOT_ZIP = Path('dataset_A.zip')

np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid', context='talk')


def configure_cjk_font():
    candidates = [
        'Noto Sans CJK SC',
        'Noto Sans CJK JP',
        'SimHei',
        'Microsoft YaHei',
        'PingFang SC',
        'WenQuanYi Zen Hei',
        'Arial Unicode MS',
    ]
    available = {f.name for f in font_manager.fontManager.ttflist}
    selected = next((f for f in candidates if f in available), None)
    if selected:
        plt.rcParams['font.family'] = selected
        print(f'使用中文字体: {selected}')
    else:
        plt.rcParams['font.family'] = 'sans-serif'
        plt.rcParams['font.sans-serif'] = ['DejaVu Sans'] + candidates
        print('未找到本机 CJK 字体，已设置多级 fallback；建议环境安装 Noto Sans CJK/SimHei。')
    plt.rcParams['axes.unicode_minus'] = False


configure_cjk_font()


## 2. 数据读取与统一预处理函数（FD001–FD004）


In [ ]:
base_cols = ['unit', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3']
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]
ALL_COLS = base_cols + sensor_cols


def load_cmapss_subset(root_zip: Path, subset: str):
    with zipfile.ZipFile(root_zip, 'r') as outer:
        inner_zip_name = [n for n in outer.namelist() if n.endswith('.zip')][0]
        inner_data = outer.read(inner_zip_name)

    with zipfile.ZipFile(io.BytesIO(inner_data), 'r') as inner:
        def read_df(file_name):
            text = inner.read(file_name).decode('utf-8', errors='ignore')
            return pd.read_csv(io.StringIO(text), sep=r'\s+', header=None)

        train = read_df(f'train_{subset}.txt').iloc[:, :len(ALL_COLS)]
        test = read_df(f'test_{subset}.txt').iloc[:, :len(ALL_COLS)]
        rul = read_df(f'RUL_{subset}.txt')

    train.columns = ALL_COLS
    test.columns = ALL_COLS

    max_cycle_train = train.groupby('unit')['cycle'].transform('max')
    train['RUL'] = max_cycle_train - train['cycle']

    max_cycle_test = test.groupby('unit', as_index=False)['cycle'].max().rename(columns={'cycle': 'max_cycle'})
    max_cycle_test['final_rul'] = rul.iloc[:, 0].values
    test = test.merge(max_cycle_test, on='unit', how='left')
    test['RUL'] = test['max_cycle'] + test['final_rul'] - test['cycle']
    test = test.drop(columns=['max_cycle', 'final_rul'])

    train['fail_within_30'] = (train['RUL'] <= N_HORIZON).astype(int)
    test['fail_within_30'] = (test['RUL'] <= N_HORIZON).astype(int)
    return train, test


def build_features(train_df: pd.DataFrame, test_df: pd.DataFrame):
    feature_candidates = ['op_setting_1', 'op_setting_2', 'op_setting_3'] + sensor_cols
    nunique = train_df[feature_candidates].nunique()
    usable_features = nunique[nunique > 1].index.tolist()

    sensor_variance = train_df[sensor_cols].var().sort_values(ascending=False)
    top_sensors = [s for s in sensor_variance.index if s in usable_features][:8]

    def add_time_features(df):
        out = df.copy().sort_values(['unit', 'cycle'])
        g = out.groupby('unit')
        for c in top_sensors:
            out[f'{c}_rollmean_5'] = g[c].transform(lambda x: x.rolling(5, min_periods=1).mean())
            out[f'{c}_rollstd_5'] = g[c].transform(lambda x: x.rolling(5, min_periods=1).std().fillna(0.0))
            out[f'{c}_lag1'] = g[c].shift(1).bfill()
        return out

    train_fe = add_time_features(train_df)
    test_fe = add_time_features(test_df)

    base_used = [c for c in usable_features if c in train_fe.columns]
    extra_feats = [
        c for c in train_fe.columns
        if c.endswith('_rollmean_5') or c.endswith('_rollstd_5') or c.endswith('_lag1')
    ]
    features = base_used + extra_feats
    return train_fe, test_fe, features, top_sensors


## 3. 统一建模：分类 vs 回归（多算法）
- 分类算法：RandomForestClassifier、LogisticRegression
- 回归算法：RandomForestRegressor、GradientBoostingRegressor
- 指标：
  - 分类：Recall、AP(AUC-PR)、ROC-AUC（并补充 Precision/F1/Brier）
  - 回归：RMSE（并补充 MAE/R²）


In [ ]:
clf_models = {
    'RandomForestClassifier': RandomForestClassifier(
        n_estimators=120,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
    ]),
}

reg_models = {
    'RandomForestRegressor': RandomForestRegressor(
        n_estimators=120,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    'GradientBoostingRegressor': GradientBoostingRegressor(random_state=RANDOM_STATE),
}

clf_results = []
reg_results = []
dataset_overview = []
best_assets = {}

for subset in SUBSETS:
    train_df, test_df = load_cmapss_subset(ROOT_ZIP, subset)
    train_fe, test_fe, FEATURES, top_sensors = build_features(train_df, test_df)

    X_train = train_fe[FEATURES]
    X_test = test_fe[FEATURES]
    y_train_cls = train_fe['fail_within_30']
    y_test_cls = test_fe['fail_within_30']
    y_train_rul = train_fe['RUL']
    y_test_rul = test_fe['RUL']

    dataset_overview.append({
        'subset': subset,
        'train_rows': len(train_df),
        'test_rows': len(test_df),
        'train_units': train_df['unit'].nunique(),
        'test_units': test_df['unit'].nunique(),
        'pos_rate_train': y_train_cls.mean(),
        'pos_rate_test': y_test_cls.mean(),
        'n_features': len(FEATURES),
    })

    trend_sensor = top_sensors[0] if top_sensors else None

    best_clf_name, best_ap = None, -np.inf
    for name, model in clf_models.items():
        model.fit(X_train, y_train_cls)
        prob = model.predict_proba(X_test)[:, 1]
        pred = (prob >= 0.5).astype(int)
        ap = average_precision_score(y_test_cls, prob)
        rec = recall_score(y_test_cls, pred)
        roc = roc_auc_score(y_test_cls, prob)

        clf_results.append({
            'subset': subset,
            'model': name,
            'Recall': rec,
            'AP(AUC-PR)': ap,
            'ROC-AUC': roc,
            'Precision': precision_score(y_test_cls, pred, zero_division=0),
            'F1': f1_score(y_test_cls, pred, zero_division=0),
            'Brier': brier_score_loss(y_test_cls, prob),
        })

        if ap > best_ap:
            best_ap = ap
            best_clf_name = name
            best_assets.setdefault(subset, {})['clf'] = {
                'name': name,
                'y_true': y_test_cls.to_numpy(),
                'y_prob': prob,
                'y_pred': pred,
                'cm': confusion_matrix(y_test_cls, pred),
            }

    best_reg_name, best_rmse = None, np.inf
    for name, model in reg_models.items():
        model.fit(X_train, y_train_rul)
        pred = model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test_rul, pred))

        reg_results.append({
            'subset': subset,
            'model': name,
            'RMSE': rmse,
            'MAE': mean_absolute_error(y_test_rul, pred),
            'R2': r2_score(y_test_rul, pred),
        })

        if rmse < best_rmse:
            best_rmse = rmse
            best_reg_name = name
            best_assets.setdefault(subset, {})['reg'] = {
                'name': name,
                'y_true': y_test_rul.to_numpy(),
                'y_pred': pred,
                'err': y_test_rul.to_numpy() - pred,
            }

    rf_imp = RandomForestClassifier(
        n_estimators=160,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rf_imp.fit(X_train, y_train_cls)
    imp = pd.Series(rf_imp.feature_importances_, index=FEATURES).sort_values(ascending=False)
    best_assets.setdefault(subset, {})['importance'] = imp

    trend_data = train_fe[['unit', 'cycle', trend_sensor]].copy() if trend_sensor else pd.DataFrame()
    if trend_sensor:
        trend_data['life_pct'] = trend_data['cycle'] / trend_data.groupby('unit')['cycle'].transform('max')
        trend_data['life_bin'] = pd.cut(trend_data['life_pct'], bins=np.linspace(0, 1, 21), include_lowest=True)
        trend_mean = trend_data.groupby('life_bin', observed=False)[trend_sensor].mean().reset_index()
        trend_mean['life_center'] = [interval.mid for interval in trend_mean['life_bin']]
    else:
        trend_mean = pd.DataFrame(columns=['life_center', 'value'])

    best_assets[subset]['trend_sensor'] = trend_sensor
    best_assets[subset]['trend_mean'] = trend_mean

    print(f"{subset}: done | features={len(FEATURES)} | best_clf={best_clf_name} | best_reg={best_reg_name}")

overview_df = pd.DataFrame(dataset_overview)
clf_results_df = pd.DataFrame(clf_results)
reg_results_df = pd.DataFrame(reg_results)


## 4. 数据集概览（FD001–FD004）


In [ ]:
display_cols = ['subset', 'train_rows', 'test_rows', 'train_units', 'test_units', 'pos_rate_train', 'pos_rate_test', 'n_features']
overview_show = overview_df[display_cols].copy()
overview_show['pos_rate_train'] = overview_show['pos_rate_train'].round(3)
overview_show['pos_rate_test'] = overview_show['pos_rate_test'].round(3)
print(overview_show)


## 5. 跨数据集比较表（Classification）


In [ ]:
clf_table = clf_results_df.copy()
clf_table = clf_table.sort_values(['subset', 'AP(AUC-PR)'], ascending=[True, False])

summary_cls = clf_table.pivot_table(
    index='subset',
    columns='model',
    values=['Recall', 'AP(AUC-PR)', 'ROC-AUC', 'Precision', 'F1', 'Brier']
).sort_index()

print('分类完整结果：')
print(clf_table.round(4))

print('按子集×模型汇总（分类）：')
print(summary_cls.round(4))


## 6. 跨数据集比较表（Regression）


In [ ]:
reg_table = reg_results_df.copy().sort_values(['subset', 'RMSE'])
summary_reg = reg_table.pivot_table(index='subset', columns='model', values=['RMSE', 'MAE', 'R2']).sort_index()

print('回归完整结果：')
print(reg_table.round(4))

print('按子集×模型汇总（回归）：')
print(summary_reg.round(4))


## 7. 专业可视化
包含：趋势分析、混淆矩阵、PR 曲线、校准曲线、特征重要性、回归误差分布。


In [ ]:
# 7.1 趋势分析：每个子集展示一个最关键传感器（方差最高）
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
axes = axes.ravel()
for i, subset in enumerate(SUBSETS):
    ax = axes[i]
    trend_sensor = best_assets[subset]['trend_sensor']
    trend_mean = best_assets[subset]['trend_mean']
    if trend_sensor is not None and not trend_mean.empty:
        ax.plot(trend_mean['life_center'], trend_mean[trend_sensor], marker='o', linewidth=2)
        ax.set_title(f'{subset} | Key Sensor Trend: {trend_sensor}')
    else:
        ax.set_title(f'{subset} | No trend sensor available')
    ax.set_xlabel('Normalized Life Cycle')
    ax.set_ylabel('Mean Sensor Value')
plt.tight_layout()
plt.show()

# 7.2 混淆矩阵：每个子集Best Classifier
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()
for i, subset in enumerate(SUBSETS):
    ax = axes[i]
    cm = best_assets[subset]['clf']['cm']
    name = best_assets[subset]['clf']['name']
    ConfusionMatrixDisplay(cm).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{subset} | Best Classifier: {name}')
plt.tight_layout()
plt.show()

# 7.3 PR 曲线：每个子集Best Classifier
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()
for i, subset in enumerate(SUBSETS):
    ax = axes[i]
    y_true = best_assets[subset]['clf']['y_true']
    y_prob = best_assets[subset]['clf']['y_prob']
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    ax.plot(recall, precision, linewidth=2, label=f'AP={ap:.3f}')
    ax.set_title(f'{subset} | PR Curve')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.legend()
plt.tight_layout()
plt.show()

# 7.4 校准曲线：每个子集Best Classifier
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()
for i, subset in enumerate(SUBSETS):
    ax = axes[i]
    y_true = best_assets[subset]['clf']['y_true']
    y_prob = best_assets[subset]['clf']['y_prob']
    frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy='quantile')
    ax.plot(mean_pred, frac_pos, marker='o', label='Model')
    ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfectly calibrated')
    ax.set_title(f'{subset} | Calibration Curve')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.legend()
plt.tight_layout()
plt.show()

# 7.5 Feature重要性：跨子集Mean Importance（RandomForestClassifier）
imp_df = pd.DataFrame({subset: best_assets[subset]['importance'] for subset in SUBSETS}).fillna(0.0)
imp_df['mean_importance'] = imp_df.mean(axis=1)
top_imp = imp_df['mean_importance'].sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(x=top_imp.values, y=top_imp.index, ax=ax, color='#4C72B0')
ax.set_title('Cross-Subset Mean Top-15 Feature Importance')
ax.set_xlabel('Mean Importance')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

# 7.6 Regression Error Distribution：每个子集最优回归器
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()
for i, subset in enumerate(SUBSETS):
    ax = axes[i]
    err = best_assets[subset]['reg']['err']
    reg_name = best_assets[subset]['reg']['name']
    sns.histplot(err, bins=40, kde=True, ax=ax, color='#4C72B0')
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.set_title(f'{subset} | Regression Error Distribution ({reg_name})')
    ax.set_xlabel('RUL True - Predicted')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()


## 8. 结论与工程建议（Professional Summary）


In [ ]:
best_cls = clf_results_df.sort_values(['subset', 'AP(AUC-PR)'], ascending=[True, False]).groupby('subset').head(1)
best_reg = reg_results_df.sort_values(['subset', 'RMSE'], ascending=[True, True]).groupby('subset').head(1)

print('每个子集最优分类器（按 AP/AUC-PR）：')
print(best_cls[['subset', 'model', 'Recall', 'AP(AUC-PR)', 'ROC-AUC', 'F1', 'Brier']].reset_index(drop=True).round(4))

print('每个子集最优回归器（按 RMSE）：')
print(best_reg[['subset', 'model', 'RMSE', 'MAE', 'R2']].reset_index(drop=True).round(4))

print('结论摘要:')
print('1) FD001–FD004 在统一流程下呈现出可比的风险识别与RUL预测能力，但不同工况复杂度导致性能差异。')
print('2) 分类建议以 AP/Recall 为主进行阈值决策，结合校准曲线评估概率可靠性。')
print('3) 回归建议同时关注 RMSE 与误差分布尾部，避免极端低估造成维护延迟风险。')
print('4) 可将跨子集稳定高重要性特征用于在线健康监测与告警规则设计。')
